# DoshaNet Training

This notebook reads prep_data.npz from a run folder and writes new artifacts to that same folder.

In [1]:
import os, json
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HANConv
from sklearn.neighbors import kneighbors_graph
import optuna

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

/Users/chinmayabharadwajhs/ml/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
RUN_DIR = ''
if not RUN_DIR:
    runs_root = 'runs'
    if os.path.isdir(runs_root):
        candidates = [os.path.join(runs_root, d) for d in os.listdir(runs_root) if d.startswith('exp_')]
        candidates = [d for d in candidates if os.path.isdir(d)]
        RUN_DIR = max(candidates, key=os.path.getmtime) if candidates else ''
print('RUN_DIR:', RUN_DIR)
assert RUN_DIR, 'Set RUN_DIR to your prep output folder'

RUN_DIR: runs/exp_20260429_132506


In [3]:
prep_path = os.path.join(RUN_DIR, 'prep_data.npz')
data = np.load(prep_path, allow_pickle=True)
X_full = data['X_full'].astype(np.float32)
y_full = data['y_full'].astype(np.int64)
feature_cols = data['feature_cols'].tolist()
dosha_names = data['dosha_names'].tolist()
train_mask = data['train_mask'].astype(bool)
test_mask = data['test_mask'].astype(bool)
NUM_CLASSES = len(dosha_names)
print('Loaded features:', len(feature_cols), 'classes:', NUM_CLASSES)

Loaded features: 29 classes: 6


In [4]:
def build_hetero_graph(X, y, feature_cols, dosha_names, k_neighbors=10):
    data = HeteroData()
    num_patients = len(X)
    num_symptoms = len(feature_cols)
    num_doshas = len(dosha_names)

    data['patient'].x = torch.tensor(X, dtype=torch.float)
    data['patient'].y = torch.tensor(y, dtype=torch.long)
    data['symptom'].x = torch.eye(num_symptoms, dtype=torch.float)
    data['dosha'].x = torch.eye(num_doshas, dtype=torch.float)

    medians = np.median(X, axis=0)
    p_idx, s_idx = [], []
    for p in range(num_patients):
        for s in range(num_symptoms):
            if X[p, s] >= medians[s]:
                p_idx.append(p)
                s_idx.append(s)
    data['patient', 'has_trait', 'symptom'].edge_index = torch.tensor([p_idx, s_idx], dtype=torch.long)

    data['patient', 'belongs_to', 'dosha'].edge_index = torch.tensor(
        [list(range(num_patients)), list(y)], dtype=torch.long
    )

    adj = kneighbors_graph(X, n_neighbors=k_neighbors, metric='cosine', include_self=False)
    rows, cols = adj.nonzero()
    data['patient', 'similar_to', 'patient'].edge_index = torch.tensor([rows, cols], dtype=torch.long)
    return data

In [5]:
class HeteroDoshaNet(torch.nn.Module):
    def __init__(self, in_channels_dict, hidden_dim, num_classes, heads=4, dropout=0.3):
        super().__init__()
        self.metadata = (
            ['patient', 'symptom', 'dosha'],
            [('patient', 'has_trait', 'symptom'),
             ('patient', 'belongs_to', 'dosha'),
             ('patient', 'similar_to', 'patient')]
        )
        self.han1 = HANConv(in_channels_dict, hidden_dim, metadata=self.metadata, heads=heads, dropout=dropout)
        self.bn = torch.nn.BatchNorm1d(hidden_dim)
        self.drop = torch.nn.Dropout(dropout)
        self.han2 = HANConv(hidden_dim, num_classes, metadata=self.metadata, heads=1, dropout=dropout)

    def forward(self, x_dict, edge_index_dict):
        out = self.han1(x_dict, edge_index_dict)
        out = {k: F.elu(self.bn(v)) for k, v in out.items()}
        out = {k: self.drop(v) for k, v in out.items()}
        out = self.han2(out, edge_index_dict)
        return F.log_softmax(out['patient'], dim=1)

def train_epoch(model, data, optimizer, mask):
    model.train()
    optimizer.zero_grad()
    out = model(data.x_dict, data.edge_index_dict)
    loss = F.nll_loss(out[mask], data['patient'].y[mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model, data, mask):
    model.eval()
    out = model(data.x_dict, data.edge_index_dict)
    pred = out[mask].argmax(dim=1)
    acc = (pred == data['patient'].y[mask]).float().mean().item()
    return acc, pred

In [6]:
OPTUNA_EPOCHS = 150
N_TRIALS = 20

def objective(trial):
    hidden = trial.suggest_categorical('hidden', [32, 64, 128])
    heads = trial.suggest_categorical('heads', [2, 4, 8])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    k_nn = trial.suggest_int('k_neighbors', 5, 25)
    wd = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

    data = build_hetero_graph(X_full, y_full, feature_cols, dosha_names, k_neighbors=k_nn).to(DEVICE)

    in_ch = {'patient': X_full.shape[1], 'symptom': len(feature_cols), 'dosha': NUM_CLASSES}
    model = HeteroDoshaNet(in_ch, hidden, NUM_CLASSES, heads, dropout).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=OPTUNA_EPOCHS)

    for _ in range(OPTUNA_EPOCHS):
        train_epoch(model, data, opt, train_mask)
        scheduler.step()

    acc, _ = evaluate(model, data, test_mask)
    return acc

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
best_params = study.best_params
print('Best params:', best_params)
print('Best trial accuracy:', round(study.best_value * 100, 2), '%')

[I 2026-04-29 13:26:15,110] A new study created in memory with name: no-name-bd7a33dc-5901-4263-986f-ba9ce1e6a926
  0%|          | 0/20 [00:00<?, ?it/s]/var/folders/df/yfx9tpqs7qq6bc8ycgcb202r0000gn/T/ipykernel_41458/3271431911.py:27: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  data['patient', 'similar_to', 'patient'].edge_index = torch.tensor([rows, cols], dtype=torch.long)
Best trial: 0. Best value: 0.320833:   5%|▌         | 1/20 [00:01<00:26,  1.37s/it]

[I 2026-04-29 13:26:16,492] Trial 0 finished with value: 0.32083332538604736 and parameters: {'hidden': 64, 'heads': 2, 'lr': 0.00013066739238053285, 'dropout': 0.4464704583099741, 'k_neighbors': 17, 'weight_decay': 0.001331121608073689}. Best is trial 0 with value: 0.32083332538604736.


Best trial: 1. Best value: 0.558333:  10%|█         | 2/20 [00:02<00:23,  1.32s/it]

[I 2026-04-29 13:26:17,781] Trial 1 finished with value: 0.5583333373069763 and parameters: {'hidden': 64, 'heads': 2, 'lr': 0.0004059611610484307, 'dropout': 0.30990257265289517, 'k_neighbors': 14, 'weight_decay': 7.476312062252303e-05}. Best is trial 1 with value: 0.5583333373069763.


Best trial: 1. Best value: 0.558333:  15%|█▌        | 3/20 [00:04<00:22,  1.35s/it]

...

[I 2026-04-29 13:26:37,402] Trial 17 finished with value: 0.875 and parameters: {'hidden': 128, 'heads': 2, 'lr': 0.0019267238240513837, 'dropout': 0.25397964073378165, 'k_neighbors': 13, 'weight_decay': 0.0031104952611167042}. Best is trial 14 with value: 0.987500011920929.


Best trial: 14. Best value: 0.9875:  95%|█████████▌| 19/20 [00:23<00:01,  1.35s/it]

[I 2026-04-29 13:26:38,709] Trial 18 finished with value: 0.987500011920929 and parameters: {'hidden': 128, 'heads': 2, 'lr': 0.004707160508372833, 'dropout': 0.157897500312229, 'k_neighbors': 7, 'weight_decay': 0.0005591555587645792}. Best is trial 14 with value: 0.987500011920929.


Best trial: 14. Best value: 0.9875: 100%|██████████| 20/20 [00:24<00:00,  1.25s/it]

[I 2026-04-29 13:26:40,090] Trial 19 finished with value: 0.9708333611488342 and parameters: {'hidden': 128, 'heads': 4, 'lr': 0.004871645879978197, 'dropout': 0.16808967868713406, 'k_neighbors': 7, 'weight_decay': 0.0005380808322182508}. Best is trial 14 with value: 0.987500011920929.
Best params: {'hidden': 128, 'heads': 2, 'lr': 0.0038484355678745806, 'dropout': 0.22113465832307672, 'k_neighbors': 9, 'weight_decay': 0.0045499802097342945}
Best trial accuracy: 98.75 %


In [7]:
EPOCHS_FINAL = 400
PATIENCE = 50

data = build_hetero_graph(
    X_full, y_full, feature_cols, dosha_names, k_neighbors=best_params['k_neighbors']
).to(DEVICE)

in_ch = {'patient': X_full.shape[1], 'symptom': len(feature_cols), 'dosha': NUM_CLASSES}
model = HeteroDoshaNet(
    in_ch,
    hidden_dim=best_params['hidden'],
    num_classes=NUM_CLASSES,
    heads=best_params['heads'],
    dropout=best_params['dropout']
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_params['lr'],
    weight_decay=best_params['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINAL)

best_test_acc = 0.0
counter = 0
best_path = os.path.join(RUN_DIR, 'best_model.pt')

for epoch in range(EPOCHS_FINAL):
    loss = train_epoch(model, data, optimizer, train_mask)

    if epoch % 10 == 0:
        train_acc, _ = evaluate(model, data, train_mask)
        test_acc, _ = evaluate(model, data, test_mask)
        print(f'Epoch {epoch:03d} | Loss {loss:.4f} | Train {train_acc*100:.1f}% | Test {test_acc*100:.1f}%')

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            counter += 10

    if counter >= PATIENCE:
        print('Early stopping at epoch', epoch)
        break

    scheduler.step()

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
print('Best test accuracy:', round(best_test_acc * 100, 2), '%')

Epoch 000 | Loss 1.8102 | Train 34.8% | Test 37.9%
Epoch 010 | Loss 0.8288 | Train 82.3% | Test 85.4%
Epoch 020 | Loss 0.4450 | Train 87.9% | Test 90.8%
Epoch 030 | Loss 0.2809 | Train 89.7% | Test 91.7%
Epoch 040 | Loss 0.2058 | Train 94.0% | Test 94.2%
Epoch 050 | Loss 0.1416 | Train 95.5% | Test 94.2%
Epoch 060 | Loss 0.1058 | Train 95.6% | Test 93.8%
Epoch 070 | Loss 0.0854 | Train 98.3% | Test 96.7%
Epoch 080 | Loss 0.0713 | Train 98.1% | Test 97.1%
Epoch 090 | Loss 0.0580 | Train 98.4% | Test 96.7%
Epoch 100 | Loss 0.0560 | Train 99.0% | Test 97.5%
Epoch 110 | Loss 0.0504 | Train 99.0% | Test 97.5%
Epoch 120 | Loss 0.0354 | Train 98.4% | Test 96.7%
Epoch 130 | Loss 0.0365 | Train 98.9% | Test 97.9%
Epoch 140 | Loss 0.0316 | Train 99.0% | Test 97.9%
Epoch 150 | Loss 0.0323 | Train 98.9% | Test 97.5%
Epoch 160 | Loss 0.0293 | Train 99.1% | Test 97.5%
Epoch 170 | Loss 0.0272 | Train 98.6% | Test 97.5%
Epoch 180 | Loss 0.0297 | Train 99.0% | Test 98.3%
Epoch 190 | Loss 0.0300 | Train

In [8]:
graph_data = {
    'x': torch.tensor(X_full, dtype=torch.float),
    'y': torch.tensor(y_full, dtype=torch.long),
    'feature_cols': feature_cols,
    'dosha_names': dosha_names,
    'k_neighbors': best_params['k_neighbors'],
    'edge_index': data['patient', 'similar_to', 'patient'].edge_index.clone()
}
graph_path = os.path.join(RUN_DIR, 'graph_data.pt')
torch.save(graph_data, graph_path)
print('Saved:', graph_path)

Saved: runs/exp_20260429_132506/graph_data.pt
